In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
np.random.seed(42)

In [2]:
df = pd.read_csv('churn.csv')
df.columns = [c.strip() for c in df.columns]
df['Churn'] = df['Churn'].str.strip()

# target (Yes -> 1, No -> 0)

y = (df['Churn'].str.lower() == 'yes').astype(int)

# features

X_raw = df.drop(columns=['Churn'])
for col in X_raw.columns:
    if X_raw[col].dtype == 'object':
        X_raw[col] = X_raw[col].str.strip()

X = pd.get_dummies(X_raw, drop_first=True)

# standardize numeric columns

num_cols = [c for c in ['tenure', 'MonthlyCharges'] if c in X.columns]
scaler = preprocessing.StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# train/test split (70/30)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# logistic Regression (no regularization)

logit_none = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
logit_none.fit(X_train, y_train)

# evaluate on test set

proba_test = logit_none.predict_proba(X_test)[:, 1]
y_pred = (proba_test >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)
prec = report['1']['precision']
rec  = report['1']['recall']

fpr, tpr, _ = roc_curve(y_test, proba_test)
auc_val = auc(fpr, tpr)

print(f"Accuracy:  {acc}")
print(f"Precision: {prec}")
print(f"Recall:    {rec}")
print(f"AUC:       {auc_val}")

Accuracy:  0.8042654028436019
Precision: 0.6561181434599156
Recall:    0.5543672014260249
AUC:       0.835858106374189


In [3]:
logit_l1 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000)

# strong -> weak regularization (small C = strong reg, large C = weak reg)

C_grid = np.logspace(-3, 2, 10)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
search = GridSearchCV(
    estimator=logit_l1,
    param_grid={'C': C_grid},
    scoring='roc_auc',
    cv=cv,
    n_jobs=1,
    refit=True
)
search.fit(X_train, y_train)


# best model & metrics

best_C = float(search.best_params_['C'])
best_alpha = 1 / best_C
best_l1 = search.best_estimator_


proba = best_l1.predict_proba(X_test)[:, 1]
yhat  = (proba >= 0.5).astype(int)


acc  = accuracy_score(y_test, yhat)
cr   = classification_report(y_test, yhat, output_dict=True)
prec = cr['1']['precision']
rec  = cr['1']['recall']


fpr, tpr, _ = roc_curve(y_test, proba)
auc_val = auc(fpr, tpr)


print(f"Best C:         {best_C}")
print(f"alpha = 1/C:    {best_alpha}")
print(f"CV AUC mean:    {search.best_score_}")
print(f"Test Accuracy:  {acc}")
print(f"Test Precision: {prec}")
print(f"Test Recall:    {rec}")
print(f"Test AUC:       {auc_val}")


# variables kept (non-zero coefficients)

coefs = best_l1.coef_.ravel()
feature_names = X_train.columns
kept_idx = np.where(coefs != 0)[0]
kept_sorted = sorted(
    [(feature_names[i], float(coefs[i])) for i in kept_idx],
    key=lambda t: abs(t[1]), reverse=True
)


print(f"=== Kept {len(kept_idx)} features ===")
print("Kept features (name, coef):")
for name, coef in kept_sorted:
    print(f"{name:40s}{coef}")

Best C:         0.1668100537200059
alpha = 1/C:    5.9948425031894095
CV AUC mean:    0.8452726717623754
Test Accuracy:  0.8037914691943128
Test Precision: 0.6553911205073996
Test Recall:    0.5525846702317291
Test AUC:       0.837358125361771
=== Kept 20 features ===
Kept features (name, coef):
Contract_Two year                       -1.1680986982159538
PhoneService_Yes                        -1.025341408668351
MonthlyCharges                          0.9106068783558382
tenure                                  -0.8248765410563774
Contract_One year                       -0.6466984886457786
TechSupport_Yes                         -0.5490642759558054
OnlineSecurity_Yes                      -0.5376631834676373
PaymentMethod_Electronic check          0.2905744086118393
PaperlessBilling_Yes                    0.27286401669170457
Dependents_Yes                          -0.25882117435632884
OnlineBackup_Yes                        -0.25754236138582876
MultipleLines_Yes                       0.15

In [4]:
max_depth = list(range(3, 13))
max_leaf_nodes = [10, 20, 30, 40, 50, 100]
min_samples_split = [2, 5, 10, 20, 50]

param_grid = [{
    'criterion': ['gini', 'entropy'],
    'max_depth': max_depth,
    'max_leaf_nodes': max_leaf_nodes,
    'min_samples_split': min_samples_split,
}]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=0
)
search.fit(X_train, y_train)

best_dt      = search.best_estimator_
best_params  = search.best_params_
best_cv_auc  = search.best_score_

proba = best_dt.predict_proba(X_test)[:, 1]
yhat  = (proba >= 0.5).astype(int)

acc   = accuracy_score(y_test, yhat)
cr    = classification_report(y_test, yhat, output_dict=True)
prec  = cr['1']['precision']
rec   = cr['1']['recall']
fpr, tpr, _ = roc_curve(y_test, proba)
auc_test    = auc(fpr, tpr)

print("=== Most effective Decision Tree (selected by CV AUC) ===")
print(f"Best hyperparameters:       {best_params}")
print(f"CV AUC (mean across folds): {best_cv_auc}")
print(f"Test Accuracy:              {acc}")
print(f"Test Precision:             {prec}")
print(f"Test Recall:                {rec}")
print(f"Test AUC:                   {auc_test}")

# variables kept (features used in splits) & importances

used_idx = set(best_dt.tree_.feature[best_dt.tree_.feature >= 0])
used_features = [X.columns[i] for i in sorted(list(used_idx))]

importances = best_dt.feature_importances_
nonzero_imp_idx = np.where(importances > 0)[0]
top_importances = sorted(
    [(X.columns[i], float(importances[i])) for i in nonzero_imp_idx],
    key=lambda t: t[1],
    reverse=True
)

print(f"\nVariables used (kept in splits): {len(used_features)}")
print("\nFeature importances:")
for name, imp in top_importances:
    print(f"{name:40s}{imp}")

=== Most effective Decision Tree (selected by CV AUC) ===
Best hyperparameters:       {'criterion': 'entropy', 'max_depth': 5, 'max_leaf_nodes': 20, 'min_samples_split': 2}
CV AUC (mean across folds): 0.8339491702516281
Test Accuracy:              0.7796208530805687
Test Precision:             0.6121495327102804
Test Recall:                0.46702317290552586
Test AUC:                   0.8250553229097262

Variables used (kept in splits): 9

Feature importances:
Contract_Two year                       0.3599348004501824
Contract_One year                       0.22866571711770892
tenure                                  0.15536394379359325
InternetService_Fiber optic             0.12397255134242825
MonthlyCharges                          0.042698306173136504
StreamingMovies_Yes                     0.03452100103053341
StreamingTV_No internet service         0.024892237461594805
PaymentMethod_Electronic check          0.024602047673840846
gender_Male                             0.005349394

In [5]:
CV = pd.Series(df.loc[X_test.index, 'MonthlyCharges'].astype(float).values, index=X_test.index) * 12

#      TP (pred=1, true=1): 0
#      FP (pred=1, true=0): 205
#      FN (pred=0, true=1): CV - 205
#      TN (pred=0, true=0): 0

def total_cost(y_true, y_pred, cv_values):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cv_values = np.asarray(cv_values)

    tp = (y_pred == 1) & (y_true == 1)
    fp = (y_pred == 1) & (y_true == 0)
    fn = (y_pred == 0) & (y_true == 1)
    tn = (y_pred == 0) & (y_true == 0)

    cost = np.zeros_like(y_true, dtype=float)
    cost[fp] = 205.0
    cost[fn] = cv_values[fn] - 205.0

    return cost.sum(), {'TP': int(tp.sum()), 'FP': int(fp.sum()), 'FN': int(fn.sum()), 'TN': int(tn.sum())}

# probabilities from previous models

proba_logit = best_l1.predict_proba(X_test)[:, 1]
proba_dt    = best_dt.predict_proba(X_test)[:, 1]

# default decision rule (threshold = 0.5)

yhat_logit_05 = (proba_logit >= 0.5).astype(int)
yhat_dt_05    = (proba_dt >= 0.5).astype(int)

cost_logit_05, conf_logit_05 = total_cost(y_test.values, yhat_logit_05, CV.values)
cost_dt_05,    conf_dt_05    = total_cost(y_test.values, yhat_dt_05,    CV.values)

print("=== Default Threshold (0.5) ===")
print(f"Logistic (L1): total cost = ${cost_logit_05}, confusion = {conf_logit_05}")
print(f"Decision Tree: total cost = ${cost_dt_05}, confusion = {conf_dt_05}")

# baselines

yhat_none = np.zeros_like(y_test.values)  # do nth (predict all not churn)
yhat_all  = np.ones_like(y_test.values)   # send to evertone (predict all churn)

cost_none, conf_none = total_cost(y_test.values, yhat_none, CV.values)
cost_all,  conf_all  = total_cost(y_test.values, yhat_all,  CV.values)

print("\n=== Baselines (for context) ===")
print(f"Do nothing (predict all not churn): total cost = ${cost_none}, confusion = {conf_none}")
print(f"Send to everyone (predict all churn): total cost = ${cost_all}, confusion = {conf_all}")

# find minimal total cost for each model

thr_grid = np.linspace(0.0, 1.0, 501)  # step ~0.002

def find_best_threshold(proba, y_true, cv_values):
    best_cost  = np.inf
    best_thr   = None
    best_conf  = None
    best_yhat  = None
    for t in thr_grid:
        yhat = (proba >= t).astype(int)
        c, conf = total_cost(y_true, yhat, cv_values)
        if c < best_cost:
            best_cost, best_thr, best_conf, best_yhat = c, t, conf, yhat
    return best_thr, best_cost, best_conf, best_yhat

thr_logit, min_cost_logit, conf_logit_best, yhat_logit_best = find_best_threshold(proba_logit, y_test.values, CV.values)
thr_dt, min_cost_dt, conf_dt_best, yhat_dt_best = find_best_threshold(proba_dt, y_test.values, CV.values)

offers_logit = int(yhat_logit_best.sum())  # number of offers sent at best threshold
offers_dt    = int(yhat_dt_best.sum())

print("\n=== Optimal Threshold (cost-minimizing) ===")
print(f"Logistic (L1): best threshold = {thr_logit}, minimal total cost = ${min_cost_logit}\nconfusion = {conf_logit_best}, offers sent = {offers_logit}")
print(f"\nDecision Tree: best threshold = {thr_dt}, minimal total cost = ${min_cost_dt}\nconfusion = {conf_dt_best}, offers sent = {offers_dt}")

delta_vs_none_logit = cost_none - min_cost_logit
delta_vs_all_logit = cost_all  - min_cost_logit
delta_vs_none_dt = cost_none - min_cost_dt
delta_vs_all_dt = cost_all  - min_cost_dt

print("\n=== Conclusions ===")
print(f"Logistic with best threshold saves ${delta_vs_none_logit} vs do nothing\nsaves ${delta_vs_all_logit} vs send to all.")
print(f"\nTree with best threshold saves ${delta_vs_none_dt} vs do nothing\nsaves ${delta_vs_all_dt} vs send to all.")

=== Default Threshold (0.5) ===
Logistic (L1): total cost = $174667.40000000002, confusion = {'TP': 310, 'FP': 163, 'FN': 251, 'TN': 1386}
Decision Tree: total cost = $236088.2, confusion = {'TP': 262, 'FP': 166, 'FN': 299, 'TN': 1383}

=== Baselines (for context) ===
Do nothing (predict all not churn): total cost = $379179.0, confusion = {'TP': 0, 'FP': 0, 'FN': 561, 'TN': 1549}
Send to everyone (predict all churn): total cost = $317545.0, confusion = {'TP': 561, 'FP': 1549, 'FN': 0, 'TN': 0}

=== Optimal Threshold (cost-minimizing) ===
Logistic (L1): best threshold = 0.264, minimal total cost = $144851.59999999998
confusion = {'TP': 445, 'FP': 430, 'FN': 116, 'TN': 1119}, offers sent = 875

Decision Tree: best threshold = 0.322, minimal total cost = $151548.6
confusion = {'TP': 407, 'FP': 373, 'FN': 154, 'TN': 1176}, offers sent = 780

=== Conclusions ===
Logistic with best threshold saves $234327.40000000002 vs do nothing
saves $172693.40000000002 vs send to all.

Tree with best thr

In [6]:
# EV rule: send offer if CV * p - 205 > 0  ⇔  p >= 205 / CV

thr_vec = 205.0 / CV.values

# logistic EV decisions

yhat_logit_ev = (proba_logit >= thr_vec).astype(int)
cost_logit_ev, conf_logit_ev = total_cost(y_test.values, yhat_logit_ev, CV.values)
offers_logit_ev = int(yhat_logit_ev.sum())

# decision tree EV decisions

yhat_dt_ev = (proba_dt >= thr_vec).astype(int)
cost_dt_ev, conf_dt_ev = total_cost(y_test.values, yhat_dt_ev, CV.values)
offers_dt_ev = int(yhat_dt_ev.sum())

# expected net value of sent offers

exp_net_logit = float(np.sum(np.maximum(0.0, CV.values * proba_logit - 205.0)))
exp_net_dt = float(np.sum(np.maximum(0.0, CV.values * proba_dt - 205.0)))

print("=== Expected Value for Logistic (L1) ===")
print(f"Total cost = ${cost_logit_ev}")
print(f"Confusion = {conf_logit_ev}")
print(f"Offers sent = {offers_logit_ev}")
print(f"Expected net value sent = ${exp_net_logit}")

print("\n=== Expected Value for Decision Tree ===")
print(f"Total cost = ${cost_dt_ev}")
print(f"Confusion = {conf_dt_ev}")
print(f"Offers sent = {offers_dt_ev}")
print(f"Expected net value sent = ${exp_net_dt}")

print("\n=== Baselines (for context) ===")
print(f"Do nothing (predict all not churn): ${cost_none}")
print(f"Confusion = {conf_none}")
print(f"Send to everyone (predict all churn): ${cost_all}, Confusion = {conf_all}")

=== Expected Value for Logistic (L1) ===
Total cost = $144539.8
Confusion = {'TP': 419, 'FP': 440, 'FN': 142, 'TN': 1109}
Offers sent = 859
Expected net value sent = $257567.79148555928

=== Expected Value for Decision Tree ===
Total cost = $142244.0
Confusion = {'TP': 424, 'FP': 460, 'FN': 137, 'TN': 1089}
Offers sent = 884
Expected net value sent = $251884.15881263747

=== Baselines (for context) ===
Do nothing (predict all not churn): $379179.0
Confusion = {'TP': 0, 'FP': 0, 'FN': 561, 'TN': 1549}
Send to everyone (predict all churn): $317545.0, Confusion = {'TP': 561, 'FP': 1549, 'FN': 0, 'TN': 0}
